# 1. Import libraries

In [2]:
import sqlite3
import pandas as pd

# 2. Load CSV file

In [3]:
df = pd.read_csv('../data/customer_features.csv')
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,...,Payment Method,Frequency of Purchases,annual_frequency,promo_dependency,purchase_scaled,previous_scaled,frequency_scaled,review_scaled,loyalty_score,value_tier
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,...,Venmo,Fortnightly,26,1.0,0.4125,0.265306,0.490196,0.24,0.359681,Low Value
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,...,Cash,Fortnightly,26,1.0,0.5500,0.020408,0.490196,0.24,0.289222,Low Value
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,...,Credit Card,Weekly,52,1.0,0.6625,0.448980,1.000000,0.24,0.636092,Medium Value
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,...,PayPal,Weekly,52,1.0,0.8750,0.979592,1.000000,0.40,0.906837,High Value
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,...,PayPal,Annually,1,1.0,0.3625,0.612245,0.000000,0.08,0.325398,Low Value


# 3. Create SQLite database

In [4]:
conn = sqlite3.connect('customer.db')

# 4. Convert CSV into SQL table

In [5]:
df.to_sql('customers', conn, if_exists='replace',index=False)

3863

# 5. SQL Queries

#### Q1. What separates high-value customers from low-value ones, and which profiles show the strongest repeat purchase behavior?

In [6]:
query1 =''' 

SELECT
    value_tier AS customer_segment,

    COUNT("Customer ID") AS total_customers,

    ROUND(AVG("Purchase Amount (USD)"),2) AS avg_spending,

    ROUND(AVG("Previous Purchases"),2) AS avg_previous_purchases,

    ROUND(AVG(loyalty_score),2) AS avg_loyalty_score,

    ROUND(AVG(promo_dependency),2) AS avg_discount_dependency,

    ROUND(AVG(annual_frequency),2) AS avg_purchase_frequency

FROM customers

GROUP BY value_tier

ORDER BY avg_spending DESC; '''

pd.read_sql(query1, conn)

,customer_segment,total_customers,avg_spending,avg_previous_purchases,avg_loyalty_score,avg_discount_dependency,avg_purchase_frequency
0,High Value,167,75.14,42.92,0.82,0.44,47.64
1,Medium Value,1654,65.64,33.94,0.57,0.43,23.14
2,Low Value,2042,53.63,16.94,0.32,0.42,10.47


#### Answer : 
High-value customers are separated by:
- Higher spending
- More previous purchases
- Stronger loyalty score
- Lower discount dependency

High-value customers show the strongest repeat purchase behavior.

### Strongest repeat purchase customer profile

In [7]:
query2 = '''
SELECT
  Gender,
  Category,
  Location,
  "Frequency of Purchases",

  COUNT("Customer ID") AS customers,

  ROUND(AVG("Previous Purchases"),2) AS avg_repeat_purchase,

  ROUND(AVG("Purchase Amount (USD)"),2) AS avg_spend,

  ROUND(AVG(loyalty_score),2) AS loyalty

FROM customers

GROUP BY
  Gender,
  Category,
  Location,
  "Frequency of Purchases"

ORDER BY avg_repeat_purchase DESC

LIMIT 10; '''

pd.read_sql(query2, conn)

,Gender,Category,Location,Frequency of Purchases,customers,avg_repeat_purchase,avg_spend,loyalty
0,Female,Accessories,Hawaii,Monthly,1,50.0,41.0,0.58
1,Female,Accessories,Maryland,Bi-Weekly,1,50.0,82.0,0.75
2,Female,Accessories,Wisconsin,Every 3 Months,1,50.0,26.0,0.52
3,Female,Footwear,Rhode Island,Quarterly,1,50.0,82.0,0.66
4,Male,Footwear,Alaska,Annually,1,50.0,89.0,0.60
5,Male,Footwear,Maryland,Bi-Weekly,1,50.0,82.0,0.79
6,Male,Footwear,Minnesota,Quarterly,1,50.0,33.0,0.47
7,Male,Outerwear,Kansas,Bi-Weekly,1,50.0,23.0,0.57
8,Male,Outerwear,Louisiana,Weekly,1,50.0,99.0,0.98
9,Male,Outerwear,South Dakota,Quarterly,1,50.0,34.0,0.51


#### Q2. Which seasons and categories are associated with lower-tenure customers versus high previous purchase counts?

In [8]:
query3 = '''

SELECT
  Season,
  Category,

  COUNT("Customer ID") AS total_customers,

  ("Previous Purchases") AS purchase_history,

  value_tier AS customer_stage 

FROM customers
WHERE customer_stage == 'Low Value'
GROUP BY
  Season,
  Category

ORDER BY purchase_history DESC; '''

pd.read_sql(query3, conn)

,Season,Category,total_customers,purchase_history,customer_stage
0,Winter,Outerwear,44,49,Low Value
1,Winter,Footwear,58,37,Low Value
2,Fall,Accessories,183,32,Low Value
3,Summer,Clothing,224,32,Low Value
4,Spring,Clothing,231,31,Low Value
5,Spring,Footwear,91,31,Low Value
6,Fall,Clothing,223,25,Low Value
7,Summer,Accessories,153,21,Low Value
8,Fall,Outerwear,44,16,Low Value
9,Winter,Accessories,157,16,Low Value


#### Q3. Which geographies signal organic demand versus discount-driven volume?

In [9]:
query4 = '''

SELECT

  Location,

  COUNT("Customer ID") AS customers,

  ROUND(SUM("Purchase Amount (USD)"),2) AS total_revenue,

  ROUND(AVG("Purchase Amount (USD)"),2) AS avg_spend,

  ROUND(AVG(loyalty_score),2) AS loyalty,

  ROUND(AVG(promo_dependency),2) AS discount_dependency,


  CASE

    WHEN AVG(loyalty_score) >= 0.5
          AND AVG(promo_dependency) < 0.5

    THEN 'Organic Demand Region'


    WHEN AVG(promo_dependency) >= 0.5

    THEN 'Discount Driven Region'


    ELSE 'Neutral Region'

  END AS region_type
FROM customers
GROUP BY Location
ORDER BY loyalty DESC; '''

pd.read_sql(query4, conn)

,Location,customers,total_revenue,avg_spend,loyalty,discount_dependency,region_type
0,Hawaii,64,3712.0,58.00,0.50,0.48,Organic Demand Region
1,Alaska,72,4867.0,67.60,0.50,0.40,Neutral Region
2,Pennsylvania,74,4926.0,66.57,0.49,0.45,Neutral Region
3,Arizona,65,4326.0,66.55,0.48,0.34,Neutral Region
4,Wyoming,71,4309.0,60.69,0.47,0.42,Neutral Region
5,Tennessee,77,4772.0,61.97,0.47,0.36,Neutral Region
6,Iowa,68,4181.0,61.49,0.47,0.51,Discount Driven Region
7,Illinois,91,5537.0,60.85,0.47,0.40,Neutral Region
8,Alabama,89,5261.0,59.11,0.47,0.40,Neutral Region
9,North Carolina,78,4742.0,60.79,0.46,0.45,Neutral Region
